**Silver to Gold: Building BI Ready Tables**

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, FloatType, DoubleType, DateType, TimestampType
from pyspark.sql import Row

In [0]:
catalog_name = "ecommerce"

In [0]:
df_products = spark.read.table(f"{catalog_name}.silver.slv_products")
df_brands = spark.read.table(f"{catalog_name}.silver.slv_brands")
df_category = spark.read.table(f"{catalog_name}.silver.slv_category")


In [0]:
df_products.createOrReplaceTempView("v_products")
df_brands.createOrReplaceTempView("v_brands")
df_category.createOrReplaceTempView("v_category")

In [0]:
display(spark.sql("select  * from v_products limit 5"))
display(spark.sql("select distinct brand_code, brand_name from v_brands"))
display(spark.sql("select distinct trim(category_code), trim(category_name) from v_category"))

In [0]:
# make sure we're in the correct catalog
spark.sql(f"USE CATALOG {catalog_name}")
          

In [0]:
%sql

-- Build brandsxcategory mapping and write Gold table
CREATE OR REPLACE TABLE gold.gld_dim_products AS

WITH brands_categories AS (
SELECT 
    b.brand_code, 
    b.brand_name, 
    c.category_code, 
    c.category_name 
FROM v_brands b 
INNER JOIN v_category c 
ON b.category_code = c.category_code
)
SELECT 
    p.product_id, 
    p.sku,
    p.brand_code, 
    COALESCE(bc.brand_name, 'Not Available') AS brand_name,
    p.category_code, 
    COALESCE(bc.category_name, 'Not Available') AS category_name,
    p.color, 
    p.size, 
    p.material, 
    p.weight_grams, 
    p.length_cm, 
    p.width_cm, 
    p.height_cm, 
    p.rating_count, 
    p.file_name, 
    p.ingest_timestamp
FROM v_products p
LEFT JOIN brands_categories bc 
ON p.brand_code = bc.brand_code;

SELECT * FROM gold.gld_dim_products LIMIT 100

In [0]:
df_customers = spark.read.table(f"{catalog_name}.silver.slv_customers")

df_customers.createOrReplaceTempView("v_customers")

display(spark.sql("select * from v_customers limit 5"))

display(spark.sql("select distinct country_code from v_customers"))

In [0]:
# PERTITION DATA INTO REGIONS
india_region = {
    "MH": "West", "GJ": "West", "RJ": "West", 
    "KA": "South", "TN": "South", "TS": "South", "AP": "South", "KL": "South", 
    "UP": "North", "WB": "North", "DL": "North"
}
# Australia states
australia_region = {
    "VIC": "SouthEast", "WA": "West", "NSW": "East", "QLD": "NorthEast"
}
# United Kingdom states
uk_region = {
    "ENG": "England", "WLS": "Wales", "SCT": "Scotland", "NIR": "Northern Ireland"
}
# United States states
us_region = {
    "MA": "Northeast", "FL": "South", "NJ": "Northeast", "CA": "West", "NY": "Northeast", "TX": "South"
}
# UAE states
uae_region = {
    "AUH": "Abu Dhabi", "DU": "Dubai", "SHJ": "Sharjah"
}
# Singapore states
singapore_region = {
    "SIN": "Singapore"
}
# Canada states
canada_region = {
    "BC": "West", "AB": "West", "ON": "East", "QC": "East", "NS": "East", "IL": "Others"
}

# Combine into a master dictionary
country_state_map = {
    "India": india_region,
    "Australia": australia_region,
    "United Kingdom": uk_region,
    "United States": us_region,
    "United Arabia Emirates": uae_region,
    "Singapore": singapore_region,
    "Canada": canada_region
}

In [0]:
country_state_map

In [0]:
rows = []
for country, state_map in country_state_map.items():
    for state_code, region in state_map.items():
        rows.append(Row(country=country, state=state_code, region=region))
rows[:10]

In [0]:
# 2. Create mapping DataFrame
df_region_mapping = spark.createDataFrame(rows)

# Optional show mapping
df_region_mapping.show(truncate=False)


In [0]:
df_customers = spark.table(f"{catalog_name}.silver.slv_customers")
display(df_customers.limit(5))

In [0]:
df_gold_customers = df_customers.join(df_region_mapping, on=["country", "state"], how="left")
df_gold_customers = df_gold_customers.fillna({"region": 'Other'})
display(df_gold_customers.limit(50))

In [0]:
# Write raw data to gold layer (catalog: ecommerce, schema: gold, table: gld_dim_customers)
df_gold_customers.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.gold.gld_dim_customers")

**Date/Calendar**


In [0]:
df_gold_date = spark.table(f"{catalog_name}.silver.slv_date")
display(df_gold_date.limit(50))

In [0]:
df_gold_date = df_gold_date.withColumn("date_id", F.date_format(F.col("date"), "yyyyMMdd").cast("int"))

# Add month name (e.g. "January", "February", etc.)
df_gold_date = df_gold_date.withColumn("month_name", F.date_format(F.col("date"), "MMMM"))

# Add is_weekend column
df_gold_date = df_gold_date.withColumn(
    "is_weekend", 
    F.when(F.col("day_name").isin("Saturday", "Sunday"), 1).otherwise(0)
)

display(df_gold_date.limit(50))

In [0]:
# Reorder columns
desired_columns_order = [
    "date_id", 
    "date",
    "year", 
    "month_name",
    "day_name", 
    "is_weekend",
    "quarter", 
    "Week", 
    "file_name", 
    "ingest_timestamp"
]

df_gold_date = df_gold_date.select(*desired_columns_order)

display(df_gold_date.limit(50))

In [0]:
# Write raw data to gold layer (catalog: ecommerce, schema: gold, table: gld_dim_date)
df_gold_date.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.gold.gld_dim_date")

In [0]:
%sql
DESCRIBE EXTENDED ecommerce.gold.gld_dim_date;